**4**&nbsp;&nbsp;&nbsp;Name your Jupyter Notebook as:

`TASK4_<your name>_<centre number>_<index number>.ipynb`

A library currently keeps paper records about its members, books and the books loaned. The library wants to create a suitable database to store the data and to allow them to run searches for specific data. The database will have three tables: a table to store data about the books, a table about the members and a table about the loans. The fields in each table are:

`Book`:

- `BookID` – unique book number, for example, 1234
- `Title` – the book title
- `Genre` – the type of book, for example, Drama, Sci-fi, Classic.

`Member`:

- `MemberNumber` – member's unique number, for example, 634
- `FamilyName` – member's family name
- `GivenName` – member's given name.

`Loan`:

- `LoanID` – the loan's unique number, for example, 12
- `MemberNumber` – the member's unique number
- `BookID` – the unique book number
- `DateLoaned` – the date that the book was taken out by the member
- `Returned` – TRUE if the book has been returned, or FALSE if it has **not** been returned.

For each of the sub-tasks 4.1 to 4.3, add a comment statement at the beginning of the code using the hash symbol '#', to indicate the sub-task the program code belongs to, for example:

```
In [1]: #Task 4.1
        Program code

        Output:
```


**Task 4.1**

Write a Python program that uses SQL code to create the database `LIBRARY` with the three tables given. Define the primary and foreign keys for each table. **[6]**


In [9]:
#Task 4.1
import sqlite3

db = sqlite3.connect("LIBRARY.db")
cursor = db.cursor()

cursor.execute("DROP TABLE IF EXISTS Book")
cursor.execute("DROP TABLE IF EXISTS Member")
cursor.execute("DROP TABLE IF EXISTS Loan")

cursor.execute("""CREATE TABLE `Book` (
                `BookID`	INTEGER,
                `Title`	TEXT,
                `Genre`	TEXT,
                PRIMARY KEY(`BookID`)
                );""")

cursor.execute("""CREATE TABLE `Member` (
                `MemberNumber`	INTEGER,
                `FamilyName`	TEXT,
                `GivenName`	TEXT,
                PRIMARY KEY(`MemberNumber`)
                );""")

cursor.execute("""CREATE TABLE `Loan` (
                `LoanID`	INTEGER,
                `MemberNumber`	INTEGER,
                `BookID`	INTEGER,
                `DateLoaned`	TEXT,
                `Returned`	TEXT,
                FOREIGN KEY(`BookID`) REFERENCES `Book`(`BookID`),
                PRIMARY KEY(`LoanID`),
                FOREIGN KEY(`MemberNumber`) REFERENCES `Member`(`MemberNumber`)
                );""")

db.commit()
db.close()

**Task 4.2**

The text files `BOOK.txt`, `MEMBER.txt` and `LOAN.txt` store the comma-separated values for each of the tables in the database.

Write a Python program to read in the data from each file and then store each item of data in the correct place in the database. **[5]**


In [10]:
#Task 4.2
import sqlite3

db = sqlite3.connect("LIBRARY.db")
cursor = db.cursor()

with open("BOOK.txt", 'r') as file:
    for line in file:
        line = line.strip('\n')
        line = line.split(',')
        BookID = line[0]
        Title = line[1]
        Genre = line[2]
        cursor.execute("""
                    INSERT INTO Book(BookID, Title, Genre)
                    VALUES(?, ?, ?)
                    """, (BookID, Title, Genre))
        

with open("MEMBER.txt", 'r') as file:
    for line in file:
        line = line.strip('\n')
        line = line.split(',')
        MemberNumber = line[0]
        FamilyName = line[1]
        GivenName = line[2]
        cursor.execute("""
                    INSERT INTO Member(MemberNumber, FamilyName, GivenName)
                    VALUES(?, ?, ?)
                    """, (MemberNumber, FamilyName, GivenName))
        
with open("LOAN.txt", 'r') as file:
    for line in file:
        line = line.strip('\n')
        line = line.split(',')
        LoanID = line[0]
        MemberNumber = line[1]
        BookID = line[2]
        DateLoaned = line[3]
        Returned = line[4]
        cursor.execute("""
                    INSERT INTO Loan(LoanID, MemberNumber, BookID, DateLoaned, Returned)
                    VALUES(?, ?, ?, ?, ?)
                    """, (LoanID, MemberNumber, BookID, DateLoaned, Returned))
        
        
db.commit()
db.close()

**Task 4.3**

Write a Python program to input a member's number and return the names of all the books that they have had out on loan, and whether each book has been returned. **[5]**

Test your program by running the application with the member number 200 **[2]**

Save your Jupyter Notebook.


In [12]:
#Task 4.3
import sqlite3

db = sqlite3.connect("LIBRARY.db")
cursor = db.cursor()

target = input("Input a member's number: ")
print()

cursor.execute("""
            SELECT Member.MemberNumber, Book.Title, Loan.Returned
            FROM Member
            JOIN Loan
            ON Member.MemberNumber = Loan.MemberNumber
            JOIN Book
            ON Loan.BookID = Book.BookID
            WHERE Member.MemberNumber = ?""",
              (target,))

results = cursor.fetchall()
print(results)
db.commit()
db.close()


Input a member's number: 200

[(200, 'Monkey puzzle', 'FALSE'), (200, 'Contemplating Camelias', 'FALSE'), (200, 'Sandy shores', 'FALSE'), (200, 'Propogation', 'TRUE')]


**Task 4.4**

Write a Python program and the necessary files to create a web application, that displays the following data, about books that have **not** yet been returned:

- member's family name
- member's given name
- book title.

The program should return an HTML document that enables the web browser to display a table with the required data.

Save your Python program as:

`TASK_4_4_<your name>_<centre number>_<index number>.py`

with any additional files / subfolders in a folder named:

`TASK_4_4_<your name>_<centre number>_<index number>` **[6]**

Run the web application.

Save the webpage output as:

`TASK_4_4_<your name>_<centre number>_<index number>.html` **[2]**


In [15]:
#Task 4.4
import flask, sqlite3

app = flask.Flask(__name__)

@app.route("/")
def index():
    db = sqlite3.connect("LIBRARY.db")
    cursor = db.cursor()
    
    cursor.execute("""
                SELECT Member.FamilyName, Member.GivenName, Book.Title
                FROM Member
                JOIN Loan
                ON Member.MemberNumber = Loan.MemberNumber
                JOIN Book
                ON Loan.BookID = Book.BookID
                WHERE Loan.Returned = 'FALSE'
                """)
    results = cursor.fetchall()
    
    db.close()
    
    return flask.render_template("index.html", results = results)

app.run(debug = True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1